# 06 — Sending tool results

After Claude requests a tool call, you need to execute the function and send the results back. This completes the tool use workflow by providing Claude with the information it requested.

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

#  Imports
from src.utils import get_client, get_model

client = get_client()
model = get_model()

In [2]:
from datetime import datetime


DATETIME_FORMAT = "%Y-%m-%d %H:%M:%S"


def get_current_datetime(date_format=DATETIME_FORMAT):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

In [3]:
from anthropic.types import ToolParam


get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time formatted according to a specified format string. Use this tool when the user asks about the current date, time, or both. Do not use this tool if the user is asking about a historical or future date — it only returns the current moment. The format string follows Python's strftime directives (e.g., '%Y-%m-%d' for ISO date, '%H:%M:%S' for time, '%Y-%m-%d %H:%M:%S' for full datetime). An empty format string will raise an error and must be avoided.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime-compatible format string that controls the structure of the returned datetime string. For example, '%Y-%m-%d' returns '2025-04-01', '%H:%M:%S' returns '14:30:00', and '%Y-%m-%d %H:%M:%S' returns '2025-04-01 14:30:00'. Must not be an empty string. Defaults to the system DATETIME_FORMAT constant if omitted.",
        "default": DATETIME_FORMAT
      }
    },
    "required": []
  },
  "input_examples": [
    { "date_format": "%Y-%m-%d" },
    { "date_format": "%H:%M:%S" },
    { "date_format": "%Y-%m-%d %H:%M:%S" },
    { "date_format": "%A, %B %d %Y" }
  ]
})

In [4]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)
response

Message(id='msg_01RqmkmH7sXA6wM1htWNaJo6', container=None, content=[ToolUseBlock(id='toolu_01169zJgnXnBu27Fv2uWicJv', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=890, output_tokens=62, server_tool_use=None, service_tier='standard'), stop_details=None)

In [5]:
messages.append({
    "role": "assistant",
    "content": response.content
})

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01169zJgnXnBu27Fv2uWicJv', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

### Running the Tool Function
When Claude responds with a tool use block, you extract the input parameters and call your function. Here's how to access the tool parameters:

In [ ]:
from anthropic.types import ToolUseBlock

tool_use_block = next(block for block in response.content if isinstance(block, ToolUseBlock))

tool_use_block.input

{'date_format': '%H:%M:%S'}

In [9]:
get_current_datetime(**tool_use_block.input)

'10:55:59'

### Tool Result Block
After running the tool function, you need to send the results back to Claude using a tool result block. This block goes inside a user message and tells Claude what happened when you executed the tool.

The tool result block has several important properties:

- `tool_use_id` - Must match the id of the ToolUse block that this ToolResult corresponds to
- `content` - Output from running your tool, serialized as a string
- `is_error` - True if an error occurred

In [10]:
tool_result = {
    "tool_use_id": tool_use_block.id,
    "type": "tool_result",
    "content": get_current_datetime(**tool_use_block.input),
    "is_error": False
}
tool_result

{'tool_use_id': 'toolu_01169zJgnXnBu27Fv2uWicJv',
 'type': 'tool_result',
 'content': '11:10:34',
 'is_error': False}

### Handling Multiple Tool Calls
Claude can request multiple tool calls in a single response. For example, if a user asks "What's 10 + 10 and what's 30 + 30?", Claude might respond with two separate ToolUse blocks.

Each tool call gets a unique ID, and you must match these IDs when sending back results. This ensures Claude knows which result corresponds to which request, even if the results arrive in a different order.

### Building the Follow-up Request
Your follow-up request to Claude must include the complete conversation history plus the new tool result. Here's the structure:

In [11]:
messages.append({
    "role": "user",
    "content": [tool_result]
})

The complete message history now contains:

- Original user message
- Assistant message with tool use block
- User message with tool result block

In [12]:
messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01169zJgnXnBu27Fv2uWicJv', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'tool_use_id': 'toolu_01169zJgnXnBu27Fv2uWicJv',
    'type': 'tool_result',
    'content': '11:10:34',
    'is_error': False}]}]

### Making the Final Request
When sending the follow-up request, you must still include the tool schema even though you're not expecting Claude to make another tool call. Claude needs the schema to understand the tool references in your conversation history.

In [14]:
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

In [15]:
response.content[0].text

'The exact time is **11:10:34**.'